In [57]:
!pip install chromadb
!pip install -U -q "google-genai"

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [ ]:
from google import genai

# El cliente de Gemini para hacer los embedding
GEMINI_API_KEY = ''
client = genai.Client(api_key=GEMINI_API_KEY)

## API de TMDB para conseguir información de películas y cast

In [ ]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = ''
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Añadir películas a la colección "movies" de ChromaDB

In [197]:
import chromadb
chroma_client = chromadb.PersistentClient(path="chroma_db")

# Función para añadir películas encontradas a la colección
# la colección es una parte de la base de datos
# hay colecciones por categorías (películas, actores, reviews...)
def add_movies_to_collection(title: str):
  """
  Busca infomación de una película a partir de su título.

  Args:
    title (str): El título de la película sobre la que buscamos información.
  """
  movies_info = get_movies_info(title)
  movies_col = chroma_client.get_or_create_collection('movies')

  # El string que recibe el LLM con los resultados
  print("Added these movies to collection ('movies'):")

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = ""
    cast = ""

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast

  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count']
    }
  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = movies_col.get(
      ids=[str(movie['id'])],
      include=[]
    )

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:
      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=movie['overview']).embeddings[0]
      ids.append(str(movie['id']))
      embeddings.append(content_embeddings.values)
      metadatas.append(get_metadata(movie, director, cast))
      documents.append(movie['overview'])

      print(f"Added {movie['title']}.")

  if len(ids) > 0:
    movies_col.add(
        ids = ids,
        embeddings = embeddings,
        metadatas = metadatas,
        documents = documents
    )

  print(f"Added {len(ids)} movies: ")

# Ejemplo de cómo usarlo junto a la búsqueda en TMDB
add_movies_to_collection("evangelion")

Added these movies to collection ('movies'):
Added 0 movies: 


## Añadir reviews a la colección "reviews" de ChromaDB

In [183]:
def add_reviews_to_collection(title: str):
  """
  Busca reviews y opiniones sobre una película en concreto.

  Args:
    title (str): El título de la película sobre la que necesitamos opiniones.
  """

  # El string que recibe el LLM con los resultados
  print("Added these reviews to collection ('reviews'):")

  movies = get_movies_info(title)
  for movie in movies['results']:
    add_reviews_from_id(movie['id'])
    

def add_reviews_from_id(movie_id):
  reviews_col = chroma_client.get_or_create_collection('reviews')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  reviews = get_movie_reviews(movie_id)

  def get_metadata(review):
    if review["author_details"]["rating"]:
      return {
          "author"  : review["author"],
          "rating"  : review["author_details"]["rating"]
      }
    else:
      return {
          "author"  : review["author"],
      }

  for review in reviews['results']:

    # Consultamos con nuestra colección
    result = reviews_col.get(
      ids=[str(review['id'])],
      include=[]
    )
    # Si no existe, la añade
    if not result['ids'] and review['content']:
      ids.append(review['id'])
      metadatas.append(get_metadata(review))

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=review['content']).embeddings[0]
      embeddings.append(content_embeddings.values)
      documents.append(review['content'])

      if len(ids) > 0:
        reviews_col.add(
          ids = ids,
          embeddings = embeddings,
          metadatas = metadatas,
          documents = documents
        )

  print(f"Added {len(ids)} reviews.")

add_reviews_to_collection('el mago de oz')


Added these reviews to collection ('reviews'):
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.
Added 0 reviews.


## Añadir actores a la colección "people" de ChromaDB

In [182]:
import json
import requests

def add_person_to_collection(name: str):
  """
  Busca información de una persona del cast de una película.

  Args:
    name (str): El nombre de la persona que el usuario está buscando.
  """
  url = f"https://api.themoviedb.org/3/search/person?query={name}&include_adult=false&language=en-US&page=1"
  r = requests.get(url, headers=TMDB_HEADERS)
  response = json.loads(r.text)

  # El string que devuelve la tool al LLM
  print("Added these people to collection ('people'): ")

  for person in response['results']:
    add_person_from_id(person['id'])

def add_person_from_id(person_id):
  people_col = chroma_client.get_or_create_collection('people')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  result = people_col.get(
    ids=[str(details['id'])],
    include=[]
  )

  if not result['ids'] and details['biography']:
    ids.append(str(details['id']))

    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    metadatas.append({'name': details['name'], 'department': details['known_for_department'], 'gender': gender})
    content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                      contents=details['biography']).embeddings[0]
    embeddings.append(content_embeddings.values)
    documents.append(details['biography'])

    if len(ids) > 0:
      people_col.add(
      ids = ids,
      embeddings = embeddings,
      metadatas = metadatas,
      documents = documents
    )

    print(f"Added {details['name']}")
  else:
    print("Person already exists in collection")

add_person_to_collection("penelope cruz")

Added these people to collection ('people'): 
Person already exists in collection
Person already exists in collection


## Ejemplo de consulta a la colección "movies" de ChromaDB

In [63]:
# Ejemplo de hacerle una pregunta a la colección
query = "película sobre coches"

# Hay que hacer un embedding porque hemos no usamos el modelo
# nativo de chromadb, sino el de gemini al meterlos en la colección
query_embedding = client.models.embed_content(model="gemini-embedding-001",
                                              contents=query).embeddings[0]
movies_col = chroma_client.get_or_create_collection('movies')

res = movies_col.query(
    query_embeddings=[query_embedding.values],
    n_results=3
)

res['metadatas'][0]

[{'vote_count': 74,
  'director': '',
  'vote_average': 7.9,
  'movie_title': 'Fast',
  'release_date': '2010-06-24',
  'cast': 'Steve Clemmons, Charlyne Yi, ',
  'popularity': 1.0202},
 {'popularity': 5.8395,
  'cast': 'Benoît Magimel, Reem Kherici, Tewfik Jallab, Sofian Khammes, Amir el Kacem, Léon Garel, Foëd Amara, Mahdi Belemlih, Alain Figlarz, Karin Martin-Prevel, Fanny Guidecoq, Yacine ben Moussa, Olivier Audibert, Alain Martin, Blandine Ruiz, Kris Filion, Jean-Claude Lagniez, Valentin Traversi, Laurent Demianoff, Cédric Monnet, Mathilde Choisy, Cyril de la Morandière, Frédéric Alhinho, Oumar Diaoure, David Genty, Franck Merenda, Stéphane Orsolani, Ibrahima Keita, Yvon Crenn, ',
  'movie_title': 'Fast Convoy',
  'vote_count': 118,
  'director': '',
  'release_date': '2016-01-20',
  'vote_average': 5.602},
 {'vote_count': 6109,
  'director': '',
  'movie_title': 'Fast X',
  'cast': 'Vin Diesel, Michelle Rodriguez, Tyrese Gibson, Ludacris, John Cena, Nathalie Emmanuel, Jordana Bre

<h2>Creacion de Agentes con Google Agent Development Kit</h2>


In [64]:
# Instalar librerías necesarias
# !pip install google-adk litellm -q

import os
import requests
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# API Key y configuración
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

print("Configuración ADK completada.")



Configuración ADK completada.


In [129]:
def query_col(query: str, collection: str):
    """
    Consulta a una colección de las disponibles (movies, reviews y people)
    con la pregunta que ha hecho el usuario, para responder con información veraz.

    Args:
        query (str): La pregunta del usuario.
        collection (str): La colección a consultar: 'movies', 'reviews' o 'people'.
    """
    # Cogemos la colección dependiendo
    movies_col = chroma_client.get_or_create_collection(collection)

    # Hay que hacer un embedding porque hemos no usamos el modelo
    # nativo de chromadb, sino el de gemini al meterlos en la colección
    query_embedding = client.models.embed_content(model="gemini-embedding-001",
                                              contents=query).embeddings[0]

    res = movies_col.query(
    query_embeddings=[query_embedding.values],
    n_results=10
    )

    texts = res["documents"][0]
    metadatas = res["metadatas"][0]
    formatted_results = []

    for i in range(0, len(texts)):
        formatted_results.append({"text": texts[i], "metadata": metadatas[i]})

    return formatted_results

In [231]:
session_service = InMemorySessionService()
APP_NAME = "cine_app"
USER_ID = "Usuario"
SESSION_ID = "user"

# Crear sesión async
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session creada: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")


Session creada: App='cine_app', User='Usuario', Session='user'


## Agente de búsqueda de información

In [232]:

researcher_agent = Agent(
    model="gemini-2.0-flash",
    name="researcher_agent",
    description="Actualiza tu base de datos con la información que pida el usuario.",
    instruction="""Eres un asistente experto en buscar información sobre películas.
    Dispones de varias herramientas para obtener información:
    - 'add_movies_to_collection': Busca información sobre una película y la añade a tu base de datos.
    - 'add_person_to_collection': Busca información sobre una personal del cast a partir de su nombre y
    la añade a tu base de datos.
    - 'add_reviews_to_collection': Busca información sobre las reviews de una película y las añade a tu base de datos.

    Cuando recibes una pregunta debes alimentar tu base de datos para que otro agente pueda consultarla.
    Harás esto:
    1. Obtienes información ya sea de películas (add_movies_to_collection), de personas del mundo del cine
    (add_person_to_collection) o de reviews y opiniones sobre una película (add_reviews_to_collection)
    2. Respondes detalladamente con lo que acabas de hacer.
    """,
    tools=[add_movies_to_collection, add_person_to_collection, add_reviews_to_collection]
)

researcher_runner = Runner(
    agent=researcher_agent,
    app_name=APP_NAME,
    session_service=session_service
)

## Agente en loop (comprobación de respuesta)

In [223]:
from google.adk.agents import LoopAgent, LlmAgent, BaseAgent, SequentialAgent
from google.adk.tools.tool_context import ToolContext

# La frase que provoca que el modelo pare
COMPLETION_PHRASE = "Se ha satisfecho la consulta del usuario."
# Estado (en sesión) donde se irá almacenando la respuesta del modelo
STATE_CURRENT_RES = "current_res"

# Tool para acabar con el loop
def exit_loop(tool_context: ToolContext):
  """Llama a esta tool *SOLO* cuando el proceso iterativo termine,
  es decir, cuando la pregunta del usuario haya sido contestada."""
  print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")
  tool_context.actions.escalate = True
  return {}


In [228]:
import asyncio

# Función para llamar al agente async
async def call_agent_async(query: str, runner: Runner):
    print(f"\n>>> User Query: {query}")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "El agente no produjo respuesta final."

    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            break

    print(f"<<< Agent Response: {final_response_text}")

In [236]:
await call_agent_async("quien es alejandro amenabar", researcher_runner)


>>> User Query: quien es alejandro amenabar
Added these people to collection ('people'): 
Added Alejandro Amenábar
<<< Agent Response: He añadido a la base de datos información sobre Alejandro Amenabar.


In [68]:
# Clase del Agente1: Investigador de cine
class AgenteInvestigadorCine:
    def __init__(self, tmdb_api_key, gemini_client, chroma_client):
        self.TMDB_HEADERS = {
            "accept": "application/json",
            "Authorization": f"Bearer {tmdb_api_key}"
        }
        self.client = gemini_client
        self.chroma_client = chroma_client

    # Buscar películas por título
    def buscar_peliculas_tmdb(self, titulo):
        url = f"https://api.themoviedb.org/3/search/movie?query={titulo}&include_adult=false&language=en-US&page=1"
        return requests.get(url, headers=self.TMDB_HEADERS).json()

    # Obtener director y cast
    def obtener_reparto_tmdb(self, movie_id):
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"
        data = requests.get(url, headers=self.TMDB_HEADERS).json()
        director = ", ".join([p['name'] for p in data.get('crew', []) if p.get('job') == "Director"])
        cast = ", ".join([p['name'] for p in data.get('cast', [])])
        return director, cast

    # Guardar películas en ChromaDB
    def guardar_peliculas_chroma(self, peliculas):
        movies_col = self.chroma_client.get_or_create_collection('movies')
        ids, embeddings, metadatas, documents = [], [], [], []

        for movie in peliculas.get('results', []):
            if not movie.get('overview'):
                continue

            # Evitar duplicados
            result = movies_col.get(ids=[str(movie['id'])], include=[])
            if result['ids']:
                continue

            director, cast = self.obtener_reparto_tmdb(movie['id'])
            emb = self.client.models.embed_content(
                model="gemini-embedding-001",
                contents=movie['overview']
            ).embeddings[0].values

            ids.append(str(movie['id']))
            embeddings.append(emb)
            metadatas.append({
                "movie_title": movie.get('title', 'Desconocido'),
                "director": director or 'Desconocido',
                "cast": cast or 'Desconocido',
                "popularity": movie.get('popularity', 0),
                "release_date": movie.get('release_date', 'Desconocida'),
                "vote_average": movie.get('vote_average', 0),
                "vote_count": movie.get('vote_count', 0)
            })
            documents.append(movie['overview'])

        if ids:
            movies_col.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=documents)

        print(f"Agregado(s) {len(ids)} película(s) a ChromaDB")

    # Consultar películas en ChromaDB
    def consultar_peliculas_chroma(self, query, n_results=5):
        emb = self.client.models.embed_content(
            model="gemini-embedding-001",
            contents=query
        ).embeddings[0].values

        movies_col = self.chroma_client.get_or_create_collection('movies')
        res = movies_col.query(query_embeddings=[emb], n_results=n_results)

        flat_metadatas = [m for sublist in res.get('metadatas', []) for m in sublist]
        peliculas_info = []
        for m in flat_metadatas:
            peliculas_info.append({
                "titulo": m.get('movie_title', 'Sin título'),
                "director": m.get('director', 'Desconocido'),
                "cast": m.get('cast', 'Desconocido'),
                "fecha_estreno": m.get('release_date', 'Desconocida'),
                "popularidad": m.get('popularity', 0),
                "vote_average": m.get('vote_average', 0),
                "vote_count": m.get('vote_count', 0)
            })
        return peliculas_info


In [69]:
# Crear instancia del agente
agente = AgenteInvestigadorCine(
    tmdb_api_key=TMDB_API_KEY,
    gemini_client=client,
    chroma_client=chroma_client
)


In [70]:
#Tool que ya existe para buscar en WikiPedia
# !pip install wikipedia -q
import wikipedia

import wikipedia

def buscar_en_wikipedia(titulo: str):
    try:
        query = f"{titulo} (film)"
        summary = wikipedia.summary(query, sentences=3)
        url = wikipedia.page(query).url
        return {"summary": summary, "url": url}
    except wikipedia.exceptions.DisambiguationError as e:
        # Si hay ambigüedad, elegir la opción que contenga "film" o "película"
        for option in e.options:
            if "film" in option.lower() or "película" in option.lower():
                summary = wikipedia.summary(option, sentences=3)
                url = wikipedia.page(option).url
                return {"summary": summary, "url": url}
        return f"No se encontró la página de la película para '{titulo}'."
    except Exception:
        return f"No se encontró información en Wikipedia para '{titulo}'."




In [76]:


# Funciones que ADK puede llamar
def buscar_y_guardar_peliculas(titulo: str):
    pelis = agente.buscar_peliculas_tmdb(titulo)
    agente.guardar_peliculas_chroma(pelis)
    return f"Se agregaron {len(pelis.get('results', []))} películas de '{titulo}' a ChromaDB."

def obtener_peliculas_por_titulo(titulo: str):
    resultados = agente.consultar_peliculas_chroma(titulo, n_results=1)
    if resultados:
        return resultados[0]
    # si no , mirar en wikipedia
    wiki = buscar_en_wikipedia(titulo)
    return wiki if isinstance(wiki, dict) else f"No se encontró información sobre '{titulo}'."

def obtener_detalles_pelicula(titulo: str):
    resultados = agente.consultar_peliculas_chroma(titulo, n_results=1)
    if resultados:
        return {
            "titulo": resultados[0]["titulo"],
            "director": resultados[0]["director"],
            "cast": resultados[0]["cast"],
            "fecha_estreno": resultados[0]["fecha_estreno"]
        }
    # Si no , mirar en wikipedia
    wiki = buscar_en_wikipedia(titulo)
    if isinstance(wiki, dict):
        return {
            "titulo": titulo,
            "summary": wiki.get("summary"),
            "url": wiki.get("url")
        }
    return f"No se encontró información sobre '{titulo}'."

def obtener_sinopsis_pelicula(titulo: str):
    resultados = agente.consultar_peliculas_chroma(titulo, n_results=1)
    if resultados:
        overview = resultados[0].get("overview")
        if overview and overview.strip():
            return overview
    # Si no hay overview o no hay resultados, mirar en Wikipedia
    wiki = buscar_en_wikipedia(titulo)
    if isinstance(wiki, dict):
        return wiki.get("summary", "Sinopsis no disponible.")
    return f"No se encontró información sobre '{titulo}'."




In [77]:
# Crear agente con ADK
agente_cine = Agent(
    name="AgenteCine",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en películas, consulta TMDB y ChromaDB",
    instruction=(
        "Eres un asistente experto en películas. "
        "Puedes buscar películas por título en TMDB y almacenarlas en ChromaDB, "
        "o consultar películas almacenadas según una descripción o título. "
        "Cuando el usuario pida buscar películas, usa 'buscar_y_guardar_peliculas'. "
        "Cuando el usuario pida consultar películas, usa 'obtener_peliculas_por_titulo' o 'obtener_detalles_pelicula'."
        "Si el usuario pregunta algo sobre el argumento de la pelicula o sus reseñas (reviews) , usa 'obtener_sinopsis_pelicula'"
        "Si el resultado obtenido contiene informacion de wikipedia , filtra y devuelve al usuario solo la informacion solicitada , por ejemplo si pregunta por un director , devuelve el director de esa pelicula ."
    ),
    tools=[buscar_y_guardar_peliculas, obtener_peliculas_por_titulo, obtener_detalles_pelicula,obtener_sinopsis_pelicula]
)

print(f"Agente '{agente_cine.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


Agente 'AgenteCine' creado con modelo 'gemini-2.0-flash'.


In [81]:
# Crear Runner y sesión
session_service = InMemorySessionService()
APP_NAME = "cine_app"
USER_ID = "Usuario"
SESSION_ID = "user"

import asyncio

# Crear sesión async
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session creada: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

runner = Runner(
    agent=researcher_agent,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner creado para el agente '{runner.agent.name}'.")


Session creada: App='cine_app', User='Usuario', Session='user'
Runner creado para el agente 'researcher_agent'.


In [82]:
# Función para llamar al agente async
async def call_agent_async(query: str):
    print(f"\n>>> User Query: {query}")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "El agente no produjo respuesta final."

    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            break

    print(f"<<< Agent Response: {final_response_text}")


In [83]:
#hablar con el agente
await call_agent_async("Busca reviews de  Ready Player One")
await call_agent_async("Dime el reparto , director y fecha de estreno  de la pelicula Ready Player One")



>>> User Query: Busca reviews de  Ready Player One
<<< Agent Response: No he encontrado reviews de Ready Player One.


>>> User Query: Dime el reparto , director y fecha de estreno  de la pelicula Ready Player One
<<< Agent Response: La película Ready Player One se estrenó el 2018-03-28, fue dirigida por Steven Spielberg y el reparto incluyó a Tye Sheridan, Olivia Cooke, Ben Mendelsohn, Lena Waithe, T.J. Miller, Simon Pegg y Mark Rylance.

